# SROIE OCR Baseline

This notebook runs the Tesseract OCR baseline on a few validated canonical SROIE receipts. It is intended for qualitative inspection only: it does not extract receipt fields or train a model.

Tesseract must be installed separately from the Python package. If it is not installed or is not on `PATH`, the notebook records and displays failure statuses instead of fabricating OCR output.

In [ ]:
import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.sroie import load_sroie_dataset
from src.ocr import run_batch_ocr

DATASET_ROOT = PROJECT_ROOT / "data" / "raw"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
PREVIEW_OUTPUT = PROCESSED_ROOT / "sroie_ocr_preview.jsonl"

dataset = load_sroie_dataset(DATASET_ROOT, strict=False)
preview_records = dataset.head(3).to_dict("records")
print(f"Validated canonical samples available: {len(dataset)}")
print(f"Receipts selected for OCR preview: {len(preview_records)}")

In [ ]:
summary = run_batch_ocr(preview_records, PREVIEW_OUTPUT)
results = [
    json.loads(line)
    for line in PREVIEW_OUTPUT.read_text(encoding="utf-8").splitlines()
]

print(f"OCR results written to: {PREVIEW_OUTPUT.relative_to(PROJECT_ROOT)}")
print(
    f"Statuses: success={summary.success}, empty={summary.empty}, "
    f"failure={summary.failure}"
)

for result in results:
    print(f"\n[{result['image_id']}] status={result['status']}")
    if result["status"] == "success":
        print(result["text"] or "No recognized text")
    else:
        print(f"{result['error_type']}: {result['error_message']}")

In [ ]:
for result in results:
    figure, axis = plt.subplots(figsize=(10, 8))
    try:
        image = Image.open(result["image_path"])
        axis.imshow(image)
    except (OSError, ValueError) as error:
        axis.text(0.5, 0.5, f"Image unavailable: {error}", ha="center", va="center")
    axis.set_title(f"{result['image_id']} | {result['status']}")
    axis.axis("off")
    plt.show()

    words = pd.DataFrame(result["words"])
    if words.empty:
        print("No word-level OCR data available.")
    else:
        display(words)